This notebook contains a small demo to compare ML hyperparameter optimisation using Optuna and GridSearchCV. A graph comparing the runtime of both techniques is generated, and the benefits of using optuna are visualised.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import plotly.express as px
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
import logging
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import plotly.express as px
import sklearn
import os
import shutil
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pickle
import optuna
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from tqdm.auto import tqdm
from sklearn.model_selection import GridSearchCV
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

In [ ]:
train_residential_apartment=pd.read_csv(r"datasets\train_residential_apartment.csv")
test_residential_apartment=pd.read_csv(r"datasets\test_residential_apartment.csv")
train_residential_house=pd.read_csv(r"datasets\train_residential_house.csv")
test_residential_house=pd.read_csv(r"datasets\test_residential_house.csv")


In [ ]:
import numpy as np
import pandas as pd
import os
import time
import pickle

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold

import xgboost as xgb


def build_preprocessor(df):
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns
    cat_cols = df.select_dtypes(include=["object"]).columns

    preprocess = ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols),

        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols)
    ])

    return preprocess


def prepare_data(train_df, test_df):
    train_df = train_df[train_df["price"] > 0].copy()
    test_df = test_df[test_df["price"] > 0].copy()

    train_df["log_price"] = np.log(train_df["price"])
    test_df["log_price"] = np.log(test_df["price"])

    X_train_full = train_df.drop(columns=["price", "log_price"])
    y_train_full = train_df["log_price"]

    X_test = test_df.drop(columns=["price", "log_price"])
    y_test = test_df["price"]  # real prices

    return X_train_full, y_train_full, X_test, y_test


def train_baseline_model(X_train_full, y_train_full):
    preprocess = build_preprocessor(X_train_full)

    model = Pipeline([
        ("preprocess", preprocess),
        ("model", xgb.XGBRegressor(
            objective="reg:squarederror",
            tree_method="hist",
            verbosity=0
        ))
    ])

    model.fit(X_train_full, y_train_full)
    return model


def evaluate_on_test(model, X_test, y_test):
    preds = np.exp(model.predict(X_test))
    rmse = float(np.sqrt(mean_squared_error(y_test, preds)))
    return rmse


In [ ]:
import optuna
from sklearn.model_selection import train_test_split

def tune_xgb_optuna(train_df, test_df, name="xgb_optuna", n_trials=30):
    X_train_full, y_train_full, X_test, y_test = prepare_data(train_df, test_df)

    # ---------------- BASELINE ----------------
    baseline_model = train_baseline_model(X_train_full, y_train_full)
    baseline_rmse = evaluate_on_test(baseline_model, X_test, y_test)
    print(f"\n[OPTUNA] BASELINE TEST RMSE: {baseline_rmse:.3f}\n")

    optuna_progress = []
    best_rmse = float("inf")
    best_model = None
    start_time = time.time()

    def objective(trial):
        nonlocal best_rmse, best_model

        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full, test_size=0.2, random_state=42
        )

        preprocess = build_preprocessor(X_tr)

        params = {
            "max_depth": trial.suggest_int("max_depth", 4, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.25),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "n_estimators": trial.suggest_int("n_estimators", 300, 2000),
            "objective": "reg:squarederror",
            "tree_method": "hist",
            "verbosity": 0,
        }

        model = Pipeline([
            ("preprocess", preprocess),
            ("model", xgb.XGBRegressor(**params))
        ])

        model.fit(X_tr, y_tr)

        preds_val = np.exp(model.predict(X_val))
        rmse_val = float(np.sqrt(mean_squared_error(np.exp(y_val), preds_val)))

        elapsed = time.time() - start_time
        optuna_progress.append((elapsed, rmse_val))

        if rmse_val < best_rmse:
            best_rmse = rmse_val
            best_model = model
            print(f"[OPTUNA][AUTO-SAVE] New BEST VAL RMSE={rmse_val:.3f}")

        return rmse_val

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)

    test_rmse = evaluate_on_test(best_model, X_test, y_test)
    print(f"\n[OPTUNA] FINAL TEST RMSE: {test_rmse:.3f}")

    os.makedirs("optuna_demo", exist_ok=True)
    model_path = f"optuna_demo/{name}_BEST.pkl"
    with open(model_path, "wb") as f:
        pickle.dump(best_model, f)

    print("\n[OPTUNA] =============== COMPARISON ===============")
    print(f"Baseline TEST RMSE : {baseline_rmse:.3f}")
    print(f"Optuna   TEST RMSE : {test_rmse:.3f}")
    print("===================================================")

    return model_path, optuna_progress, baseline_rmse, test_rmse


In [ ]:
from sklearn.model_selection import ParameterGrid

def manual_gridsearch_xgb(train_df, test_df, save_name="xgb_grid", n_splits=3):
    X_train_full, y_train_full, X_test, y_test = prepare_data(train_df, test_df)

    # ---------------- BASELINE ----------------
    baseline_model = train_baseline_model(X_train_full, y_train_full)
    baseline_rmse = evaluate_on_test(baseline_model, X_test, y_test)
    print(f"\n[GRID] BASELINE TEST RMSE: {baseline_rmse:.3f}\n")

    # Param grid (small so the demo doesn't take forever)
    param_grid = {
        "max_depth": [4, 6, 8, 10],
        "learning_rate": [0.02, 0.1, 0.25],
        "subsample": [0.6, 1.0],
        "colsample_bytree": [0.6, 1.0],
        "min_child_weight": [1, 3, 7, 10],
        "n_estimators": [300, 800,1500,2000],
    }

    grid = list(ParameterGrid(param_grid))
    print(f"[GRID] Number of hyperparameter combinations: {len(grid)}")

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    best_rmse = float("inf")
    best_params = None
    gridsearch_progress = []

    start_time = time.time()

    # --------- MANUAL GRID SEARCH + CV ---------
    for i, params in enumerate(grid, start=1):
        fold_rmses = []

        for train_idx, val_idx in kf.split(X_train_full):
            X_tr = X_train_full.iloc[train_idx]
            X_val = X_train_full.iloc[val_idx]
            y_tr = y_train_full.iloc[train_idx]
            y_val = y_train_full.iloc[val_idx]

            preprocess = build_preprocessor(X_tr)

            model = Pipeline([
                ("preprocess", preprocess),
                ("model", xgb.XGBRegressor(
                    objective="reg:squarederror",
                    tree_method="hist",
                    verbosity=0,
                    **params
                ))
            ])

            model.fit(X_tr, y_tr)

            # price RMSE (not logs)
            preds_val = np.exp(model.predict(X_val))
            rmse_val = float(np.sqrt(mean_squared_error(np.exp(y_val), preds_val)))
            fold_rmses.append(rmse_val)

        mean_rmse = float(np.mean(fold_rmses))
        elapsed = time.time() - start_time
        gridsearch_progress.append((elapsed, mean_rmse))

        print(f"[GRID] Combo {i}/{len(grid)} | params={params} | VAL_RMSE={mean_rmse:.3f}")

        if mean_rmse < best_rmse:
            best_rmse = mean_rmse
            best_params = params
            print(f"[GRID][BEST UPDATE] New best VAL_RMSE={best_rmse:.3f}")

    print("\n[GRID] Best params (by VAL RMSE):")
    print(best_params)
    print(f"[GRID] Best VAL RMSE: {best_rmse:.3f}")

    # Training the final model
    preprocess_full = build_preprocessor(X_train_full)
    best_model = Pipeline([
        ("preprocess", preprocess_full),
        ("model", xgb.XGBRegressor(
            objective="reg:squarederror",
            tree_method="hist",
            verbosity=0,
            **best_params
        ))
    ])
    best_model.fit(X_train_full, y_train_full)

    test_rmse = evaluate_on_test(best_model, X_test, y_test)
    print(f"\n[GRID] FINAL TEST RMSE: {test_rmse:.3f}")

    os.makedirs("gridsearch_models", exist_ok=True)
    model_path = f"gridsearch_models/{save_name}_BEST.pkl"
    with open(model_path, "wb") as f:
        pickle.dump(best_model, f)

    print("\n[GRID] =============== COMPARISON ===============")
    print(f"Baseline TEST RMSE : {baseline_rmse:.3f}")
    print(f"GridSearch TEST RMSE: {test_rmse:.3f}")
    print("=================================================")

    return model_path, gridsearch_progress, baseline_rmse, test_rmse


In [ ]:
import os
if os.path.exists("xgb_res_apt.db"):
    os.remove("xgb_res_apt.db")


In [ ]:
# Optuna 
opt_model_path, optuna_progress, opt_base_rmse, opt_test_rmse = tune_xgb_optuna(
    train_residential_apartment,
    test_residential_apartment,
    name="xgb_res_apt_optuna",
    n_trials=50  
)

# Grid Search
grid_model_path, gridsearch_progress, grid_base_rmse, grid_test_rmse = manual_gridsearch_xgb(
    train_residential_apartment,
    test_residential_apartment,
    save_name="xgb_res_apt_grid",
    n_splits=3
)


In [ ]:
import matplotlib.pyplot as plt

opt_t, opt_rmse = zip(*optuna_progress)
gs_t, gs_rmse   = zip(*gridsearch_progress)

plt.figure(figsize=(10,5))
plt.plot(opt_t, opt_rmse, label="Optuna (VAL RMSE)", marker="o")
plt.plot(gs_t, gs_rmse, label="Manual Grid Search (VAL RMSE)", marker="o")

plt.xlabel("Time [s]")
plt.ylabel("Validation RMSE (price space)")
plt.title("Convergence speed: Optuna vs Grid Search")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()
